# 📊 Personal Finance Analysis

This project analyzes personal financial transactions to understand spending patterns, income behavior, and financial habits over time.

## 📁 Data Source

Dataset: Personal Finance Dataset (Kaggle)

The dataset contains individual financial transactions, including:
- Date
- Description
- Amount
- Transaction Type (credit/debit)
- Category
- Account Name

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_csv("../data/raw/personal_transactions.csv")


In [ ]:
df.shape
df.head()
df.dtypes
df.isna().sum()

## 🔍 Initial Observations

- The dataset contains no missing values.
- The "Date" column is initially stored as a string and needs conversion.
- The "Amount" column is already in numeric format.
- Categories appear to be well standardized.
- Transaction types are divided into "credit" and "debit".

In [ ]:
df["date"] = pd.to_datetime(df["Date"])

## 🧠 Transaction Classification

To enable more accurate financial analysis, transactions were classified into:

- Income: real money entering the account (e.g., Paycheck)
- Expense: money leaving the account (debit transactions)
- Transfer: internal movements (e.g., Credit Card Payment)
- Other: any remaining credit transactions not classified above

In [ ]:
conditions = [
    df["Category"] == "Credit Card Payment",
    df["Transaction Type"] == "debit",
    (df["Transaction Type"] == "credit") & (df["Category"] == "Paycheck")
]

choices = [
    "transfer",
    "expense",
    "income"
]

df["transaction_class"] = np.select(conditions, choices, default="other")

In [ ]:
df["transaction_class"].value_counts()

## 📆 Time Features

To analyze spending patterns over time, we extract temporal features such as year, month, and day of the week from the transaction date.


In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()
df["year_month"] = df["date"].dt.to_period("M")

In [ ]:
df.head()

## 💸 Expenses by Category

In this section, we analyze how expenses are distributed across different categories to identify the main cost drivers.

In [ ]:
df_expenses_category = (
    df[df["transaction_class"] == "expense"]
    .groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print(df_expenses_category)

## 📊 Expense Distribution by Category

This chart shows how expenses are distributed across categories, helping identify the main cost drivers.

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df_expenses_category,
    y="Category",
    x="Amount"
)

plt.title("Expenses by Category", fontsize=14, pad=12)
plt.xlabel("Amount")
plt.ylabel("Category")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("../images/expenses_category.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df["transaction_class"].value_counts()

### 🔍 Key Insights

- Housing costs (Mortgage & Rent) represent the largest share of expenses, indicating a high fixed cost structure in the budget.

- Home Improvement stands out as the second largest category, suggesting either a period of elevated spending (e.g., renovations) or potential seasonality. Further time-based analysis is needed to confirm whether this is recurring or sporadic.

- Essential living expenses such as Groceries, Utilities, and Restaurants show similar spending levels, indicating a relatively stable pattern in day-to-day consumption.

- There is a long tail of smaller expense categories, which individually have low impact but may represent a meaningful portion of total spending when combined.

## 📈 Monthly Expense Trend

In this section, we analyze how expenses evolve over time to identify trends, fluctuations, and possible seasonality patterns.

In [ ]:
df_monthly_expenses = (
    df[df["transaction_class"] == "expense"]
    .groupby("year_month")["Amount"]
    .sum()
    .reset_index()
    .sort_values("year_month")
)

print(df_monthly_expenses)

## 📊 Monthly Expense Trend

This chart shows how total expenses evolve over time, allowing us to identify trends, fluctuations, and potential seasonality patterns.

In [ ]:
df_monthly_expenses["year_month"] = df_monthly_expenses["year_month"].astype(str)


plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df_monthly_expenses,
    x="year_month",
    y="Amount"
)

plt.title("Monthly Expenses Over Time", fontsize=14, pad=12)
plt.xlabel("Month")
plt.ylabel("Amount")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig("../images/monthly_expenses.png", dpi=300, bbox_inches="tight")
plt.show()

### 📈 Monthly Expense Insights

- Overall, expenses show a relatively stable pattern across most months, typically ranging between 2,000 and 2,500.

- Two significant spending spikes are observed (around mid-2018 and mid-2019), indicating the presence of non-recurring or extraordinary expenses, such as renovations or travel.

- A slight increase in spending is noticeable towards the end of the year (November and December), suggesting potential seasonality effects related to holidays such as Christmas and New Year.

- Aside from these peaks, the financial behavior remains consistent, reinforcing the idea of a controlled and predictable expense structure.

## 📈 Monthly Income Trend

This section analyzes how income evolves over time, helping identify patterns and consistency in earnings.

In [ ]:
df_monthly_income = (
    df[df["transaction_class"] == "income"]
    .groupby("year_month")["Amount"]
    .sum()
    .reset_index()
    .sort_values("year_month")
)
print(df_monthly_income)

## ⚖️ Monthly Balance Calculation

In this section, we combine monthly income and expenses to calculate the net balance over time.

This allows us to evaluate whether there is a surplus or deficit in each month, providing a clearer view of financial sustainability and overall financial health.

In [ ]:
df_monthly_income["year_month"] = df_monthly_income["year_month"].astype(str)
df_monthly_expenses["year_month"] = df_monthly_expenses["year_month"].astype(str)

df_monthly = pd.merge(
    df_monthly_income,
    df_monthly_expenses,
    on="year_month",
    how="outer"
)

df_monthly = df_monthly.rename(columns={
    "Amount_x": "income",
    "Amount_y": "expense"
})

df_monthly["balance"] = df_monthly["income"] - df_monthly["expense"]

df_monthly.head()

## 📊 Monthly Balance Trend

This chart shows the net financial balance over time, highlighting periods of surplus and deficit.

It helps identify financial stability, potential risks, and months where expenses exceeded income.

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df_monthly, 
    x="year_month",
    y="balance" 
)

plt.title("Monthly Balance Over Time", fontsize=14, pad=12) 
plt.xlabel("Month")
plt.ylabel("Balance") 
plt.axhline(0, color="red", linestyle="--", linewidth=1)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()

plt.savefig("../images/monthly_balance.png", dpi=300, bbox_inches="tight") 
plt.show()

### 💰 Monthly Balance Insights

- Overall, the financial balance remains positive in most months, indicating a generally sustainable financial situation with income consistently exceeding expenses.

- A significant negative balance is observed around mid-2018, highlighting a period where expenses largely exceeded income. This aligns with previously identified expense spikes and suggests the presence of extraordinary spending events.

- Aside from this outlier, the balance shows relative stability, reinforcing the idea of controlled and predictable financial behavior over time.

- Income appears consistent across months, while fluctuations in balance are primarily driven by changes in expenses rather than earnings.

- The presence of occasional negative or low-balance months suggests potential financial risk periods, emphasizing the importance of monitoring irregular high expenses.